# Guess-the-Song: vocal-removed audio (Kaggle, dual T4)

Produces only the instrumental (vocals removed) version of each song, via Mel-Band Roformer (Kim FT2 Bleedless checkpoint). No Level 1/3, no silence trimming, no intermediate wav files kept on disk — each song is converted mp3->wav, run through the model, written straight to mp3, and the wav is deleted before moving to the next song.

**Before running:**
1. Upload `music_collection_for_colab.zip` as a Kaggle Dataset and attach it via **+ Add Input** (Kaggle auto-extracts it, so mp3s land directly under `/kaggle/input/<slug>/`).
2. **Settings > Accelerator > GPU T4 x2**.

Real dual-GPU use here means two workers running concurrently on separate GPUs, each churning through half the songs sequentially — not `DataParallel`, which can't help when each inference call is already batch-size-1 (see below).

In [ ]:
import subprocess, os, glob, shutil, threading
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'melband-roformer-infer'])
subprocess.run(['apt-get', '-qq', '-y', 'install', 'ffmpeg'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch
import yaml
import numpy as np
import soundfile as sf
from ml_collections import ConfigDict
from mel_band_roformer.model_registry import MODEL_REGISTRY
from mel_band_roformer.download import download_model_assets
from mel_band_roformer.utils import get_model_from_config, demix_track

# ---------------------------------------------------------------
# 1. Locate input mp3s (Kaggle auto-extracts a zip dataset; the
#    zip fallback below is only for the rare case it didn't).
# ---------------------------------------------------------------
INPUT_ROOT = '/kaggle/input'
WORK_DIR = '/kaggle/working'

mp3_files = sorted(glob.glob(os.path.join(INPUT_ROOT, '**', '*.mp3'), recursive=True))
if not mp3_files:
    zips = glob.glob(os.path.join(INPUT_ROOT, '**', '*.zip'), recursive=True)
    if not zips:
        raise FileNotFoundError('No mp3s or zip found under /kaggle/input -- attach the dataset.')
    raw_dir = os.path.join(WORK_DIR, 'raw_mp3')
    os.makedirs(raw_dir, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(raw_dir)
    mp3_files = sorted(glob.glob(os.path.join(raw_dir, '**', '*.mp3'), recursive=True))

song_ids = [os.path.splitext(os.path.basename(p))[0] for p in mp3_files]
print(f'Found {len(mp3_files)} songs')

# ---------------------------------------------------------------
# 2. Download the checkpoint and patch two upstream config bugs:
#    1) it's dumped with a `!!python/tuple` YAML tag, which
#       `yaml.safe_load` refuses to construct for security reasons.
#    2) chunk_size lives under 'audio:' but inference code reads it
#       from 'inference:' -> KeyError without this patch.
# ---------------------------------------------------------------
MODEL_SLUG = 'melband-roformer-kimmel-ft2-bleedless'
model_dir = Path(WORK_DIR) / 'models'
download_model_assets([MODEL_REGISTRY.get(MODEL_SLUG)], model_dir)

config_path = model_dir / MODEL_SLUG / 'config_kimmel_unwa_ft.yaml'
model_path = model_dir / MODEL_SLUG / 'kimmel_unwa_ft2_bleedless.ckpt'

raw_yaml = config_path.read_text().replace('!!python/tuple', '')
cfg_dict = yaml.safe_load(raw_yaml)
if 'chunk_size' not in cfg_dict.get('inference', {}):
    cfg_dict.setdefault('inference', {})['chunk_size'] = cfg_dict['audio']['chunk_size']
config = ConfigDict(cfg_dict)

target_instr = config.training.target_instrument or config.training.instruments[0]
state_dict = torch.load(model_path, map_location='cpu')  # loaded once, read-only, shared by both workers

n_gpus = torch.cuda.device_count()
print(f'GPUs visible: {n_gpus}')
for i in range(n_gpus):
    print(f'  cuda:{i} -> {torch.cuda.get_device_name(i)}')

# ---------------------------------------------------------------
# 3. Per-song pipeline: mp3 -> temp wav -> model -> instrumental
#    mp3 -> delete temp wav. Never writes vocals or keeps any wav.
# ---------------------------------------------------------------
FINAL_DIR = Path(WORK_DIR) / 'final_output'
FINAL_DIR.mkdir(parents=True, exist_ok=True)

failed = []
failed_lock = threading.Lock()

def process_one(mp3_path, song_id, model, device, temp_wav, temp_out_wav):
    subprocess.run(['ffmpeg', '-y', '-i', mp3_path, '-ar', '44100', str(temp_wav)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        mix, sr = sf.read(str(temp_wav))
        original_mono = False
        if len(mix.shape) == 1:
            original_mono = True
            mix = np.stack([mix, mix], axis=-1)
        mixture = torch.tensor(mix.T, dtype=torch.float32)

        res, _ = demix_track(config, model, mixture, device)

        vocals_output = res[target_instr].T
        if original_mono:
            vocals_output = vocals_output[:, 0]

        instrumental = mix - vocals_output
        sf.write(str(temp_out_wav), instrumental, sr, subtype='FLOAT')

        out_path = FINAL_DIR / f'{song_id}.mp3'
        subprocess.run(['ffmpeg', '-y', '-i', str(temp_out_wav), str(out_path)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    finally:
        temp_wav.unlink(missing_ok=True)
        temp_out_wav.unlink(missing_ok=True)

def worker(gpu_id, songs, worker_name):
    device = torch.device(f'cuda:{gpu_id}' if n_gpus > 0 else 'cpu')
    model = get_model_from_config('mel_band_roformer', config)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    temp_wav = Path(WORK_DIR) / f'_temp_{worker_name}.wav'
    temp_out_wav = Path(WORK_DIR) / f'_temp_{worker_name}_out.wav'

    for i, (mp3_path, song_id) in enumerate(songs, 1):
        try:
            process_one(mp3_path, song_id, model, device, temp_wav, temp_out_wav)
        except Exception as e:
            print(f'[{worker_name} {i}/{len(songs)}] FAILED {song_id}: {e}')
            with failed_lock:
                failed.append(song_id)
        if i % 25 == 0 or i == len(songs):
            print(f'[{worker_name}] {i}/{len(songs)} done')

songs = list(zip(mp3_files, song_ids))

if n_gpus >= 2:
    half = len(songs) // 2
    t0 = threading.Thread(target=worker, args=(0, songs[:half], 'gpu0'))
    t1 = threading.Thread(target=worker, args=(1, songs[half:], 'gpu1'))
    t0.start(); t1.start()
    t0.join(); t1.join()
else:
    print('Fewer than 2 GPUs visible -- set Settings > Accelerator > GPU T4 x2. '
          'Falling back to a single worker.')
    worker(0, songs, 'gpu0')

print(f'\nDone. Failed: {len(failed)} -> {failed}')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/guess_the_song_instrumentals', 'zip', str(FINAL_DIR))
print('Saved to /kaggle/working/guess_the_song_instrumentals.zip')

## Drums-only version (htdemucs_ft's drums-specialized sub-model)

Fully independent of the vocal-removal section above -- runs standalone with its own imports, input discovery, and GPU check, so you can run just this section (e.g. after a kernel restart, or without ever running the vocals cells).

Same approach as the vocal-removal section: two worker threads, one per GPU, each processing its half of the songs sequentially (mp3 -> wav -> model -> drums mp3 -> delete wav -> next). Uses `f7e0c4bc`, the drums-specialized checkpoint from htdemucs_ft's 4-model bag (see the markdown two cells up in the chat for why), at the same one-model compute cost as base htdemucs.

In [ ]:
import os, glob, zipfile, subprocess, threading
from pathlib import Path

subprocess.run(['pip', 'install', '-q', 'demucs'])
subprocess.run(['apt-get', '-qq', '-y', 'install', 'ffmpeg'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch
import numpy as np
import soundfile as sf
from demucs.pretrained import get_model
from demucs.apply import apply_model

# ---------------------------------------------------------------
# 1. Locate input mp3s (Kaggle auto-extracts a zip dataset; the
#    zip fallback below is only for the rare case it didn't).
# ---------------------------------------------------------------
INPUT_ROOT = '/kaggle/input'
WORK_DIR = '/kaggle/working'

mp3_files = sorted(glob.glob(os.path.join(INPUT_ROOT, '**', '*.mp3'), recursive=True))
if not mp3_files:
    zips = glob.glob(os.path.join(INPUT_ROOT, '**', '*.zip'), recursive=True)
    if not zips:
        raise FileNotFoundError('No mp3s or zip found under /kaggle/input -- attach the dataset.')
    raw_dir = os.path.join(WORK_DIR, 'raw_mp3')
    os.makedirs(raw_dir, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(raw_dir)
    mp3_files = sorted(glob.glob(os.path.join(raw_dir, '**', '*.mp3'), recursive=True))

song_ids = [os.path.splitext(os.path.basename(p))[0] for p in mp3_files]
songs = list(zip(mp3_files, song_ids))
print(f'Found {len(mp3_files)} songs')

# ---------------------------------------------------------------
# 2. GPU check
# ---------------------------------------------------------------
n_gpus = torch.cuda.device_count()
print(f'GPUs visible: {n_gpus}')
for i in range(n_gpus):
    print(f'  cuda:{i} -> {torch.cuda.get_device_name(i)}')
if n_gpus < 2:
    print('WARNING: fewer than 2 GPUs detected. Set Settings > Accelerator > GPU T4 x2, '
          'then Save & re-run. Falling back to a single worker.')

# ---------------------------------------------------------------
# 3. Per-song pipeline: mp3 -> temp wav -> model -> drums mp3 ->
#    delete temp wavs. Never keeps bass/other/vocals or any wav.
# ---------------------------------------------------------------
DRUMS_DIR = Path(WORK_DIR) / 'final_output_drums'
DRUMS_DIR.mkdir(parents=True, exist_ok=True)

drums_failed = []
drums_failed_lock = threading.Lock()

def process_one_drums(mp3_path, song_id, model, device, temp_wav):
    # htdemucs expects 44.1kHz; source clips are 48kHz, so force resample here too.
    subprocess.run(['ffmpeg', '-y', '-i', mp3_path, '-ar', str(model.samplerate), str(temp_wav)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    temp_drums_wav = temp_wav.with_name(temp_wav.stem + '_drums.wav')
    try:
        mix, sr = sf.read(str(temp_wav))
        if len(mix.shape) == 1:
            mix = np.stack([mix, mix], axis=-1)
        mixture = torch.tensor(mix.T, dtype=torch.float32)

        with torch.no_grad():
            out = apply_model(model, mixture.unsqueeze(0), device=device, progress=False)

        drums_idx = model.sources.index('drums')
        drums = out[0, drums_idx].cpu().numpy().T

        sf.write(str(temp_drums_wav), drums, sr, subtype='FLOAT')
        out_path = DRUMS_DIR / f'{song_id}.mp3'
        subprocess.run(['ffmpeg', '-y', '-i', str(temp_drums_wav), str(out_path)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    finally:
        temp_wav.unlink(missing_ok=True)
        temp_drums_wav.unlink(missing_ok=True)

def drums_worker(gpu_id, worker_songs, worker_name):
    device = torch.device(f'cuda:{gpu_id}' if n_gpus > 0 else 'cpu')
    # 'f7e0c4bc' is htdemucs_ft's drums-specialized sub-model (htdemucs_ft is a
    # bag of 4 fine-tuned models, one per stem, mixed with a diagonal weight
    # matrix -- this is the one weighted 1.0 for drums). Loading it directly
    # gives htdemucs_ft's fine-tuned drums quality for the cost of one model
    # (same as base htdemucs), instead of running all 4 sub-models.
    model = get_model('f7e0c4bc').to(device)
    model.eval()
    temp_wav = Path(WORK_DIR) / f'_temp_drums_{worker_name}.wav'

    for i, (mp3_path, song_id) in enumerate(worker_songs, 1):
        try:
            process_one_drums(mp3_path, song_id, model, device, temp_wav)
        except Exception as e:
            print(f'[{worker_name} {i}/{len(worker_songs)}] FAILED {song_id}: {e}')
            with drums_failed_lock:
                drums_failed.append(song_id)
        if i % 25 == 0 or i == len(worker_songs):
            print(f'[{worker_name}] {i}/{len(worker_songs)} done')

if n_gpus >= 2:
    half = len(songs) // 2
    t0 = threading.Thread(target=drums_worker, args=(0, songs[:half], 'gpu0'))
    t1 = threading.Thread(target=drums_worker, args=(1, songs[half:], 'gpu1'))
    t0.start(); t1.start()
    t0.join(); t1.join()
else:
    drums_worker(0, songs, 'gpu0')

print(f'\nDone. Failed: {len(drums_failed)} -> {drums_failed}')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/guess_the_song_drums', 'zip', str(DRUMS_DIR))
print('Saved to /kaggle/working/guess_the_song_drums.zip')